# MoganBERT pilot: is 8K-embed easy to transfer because of its lineage?

**Why.** 8K-embed (Qwen3-8B distilled into a 150M ModernBERT, then contrastive fine-tuning) works as a teacher (alone 65.67, with BGE-M3 67.16), while Qwen3-8B itself did not (alone 63.09). 8K-embed shares the student's tokenizer (identical 50,008-token vocabulary) and lab, so part of its transferability could come from lineage rather than from being a small Qwen-distilled model.

**Control.** MoganBERT-Embed (moganai, Apache-2.0): same architecture (ModernBERT-base, 22 layers, 768-d, 8192 context, RoPE 160K/10K), same kind of recipe (Qwen3-Embedding-8B distillation + contrastive fine-tuning + checkpoint average), mean pooling + Normalize, no prompt. Different: its own from-scratch base MoganBERT-TR and its own Unigram tokenizer (50,048 tokens, only 4,029 shared with the student). So the comparison separates *lineage (tokenizer, base, lab)*, not architecture.

**Steps** (A100; re-running resumes and skips finished parts):
1. Encode MoganBERT-Embed on the 200K mix (plain). The BGE and 8K caches exist.
2. Geometry screen: BGE, 8K and MoganBERT on the first 10,000 texts.
3. Two runs (200K mix, lr 1e-4, 5400 steps, seed 42), only the teacher changes:

| | teacher | reference (same recipe) |
|---|---|---|
| A | 8K-embed alone | 65.67 (done) |
| B | `mogan_single_s5400`: MoganBERT alone | |
| C | BGE-M3 + 8K-embed | 67.16, seed 43 67.25 (done) |
| D | `two_bge_mogan_s5400`: BGE-M3 + MoganBERT | |

**Also needed:** MoganBERT-Embed's own TR-MTEB score with our protocol (preset `moganbert_embed` in `eval_trio/eval1.ipynb`, separate runtime). Read the student scores next to the teacher scores (transfer efficiency), and only differences above 0.3.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)


In [ ]:
# --- 1. Encode MoganBERT-Embed on the 200K mix (resumable at shard level) ---
!python -m turkish_distill.encode_teachers --config configs/two_bge_moganbert.yaml --status
!python -m turkish_distill.encode_teachers --config configs/two_bge_moganbert.yaml --only moganbert_embed


In [ ]:
# --- 2. Geometry screen: BGE-M3, 8K-embed and MoganBERT-Embed (first 10,000 texts of the 200K mix) ---
# Per teacher pair: Gram Pearson / Spearman, top-10 neighbour overlap, rank agreement on random pairs of pairs
# (conflict_rate = both decisive beyond the hinge margin 0.015 and in opposite order), and the same per mix bucket.
# ~1 min on CPU. Needs the repo from after this cell was added: re-run the Setup cell if the runtime is older.
os.makedirs(f'{DRIVE_ROOT}/diag', exist_ok=True)
!python -m turkish_distill.teacher_compare --config configs/three_bge_8k_moganbert.yaml --n 10000 --group-column bucket --out {DRIVE_ROOT}/diag/geometry_moganbert_mix200k.json


In [ ]:
# --- 3. Training runs ---
# Runs of this notebook, one after another. (name, config, extra overrides). A later override wins.
# Both configs already hold the recipe (200K mix, lr 1e-4, 5400 steps).
BASE = []
RUNS = [
    ('mogan_single_s5400', 'configs/single_moganbert.yaml', []),
    ('two_bge_mogan_s5400', 'configs/two_bge_moganbert.yaml', []),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        print(f"    'd_{name}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final'),")


In [ ]:
from google.colab import runtime
runtime.unassign()